# 01_task1 — PA4 Task 1 (compression) on Colab L4

Run top-to-bottom. Every code cell prints `PASS` or `FAIL`. Results go to Drive
(`/content/drive/MyDrive/atml_pa4_results/task1`), never to the repo.
The driver is resumable: re-running cell 6 after a disconnect skips finished conditions.

| Cell | What it does |
|---|---|
| 1 | Mount Drive, set HF_HOME, check the locked data files |
| 2 | Clone / pull the repo |
| 3 | Install pinned transformers / huggingface_hub |
| 4 | CPU unit tests |
| 5 | GPU sanity: `fp16` condition only |
| 6 | Overnight: the 14 non-adaptive conditions |
| 7 | P3/Q3: `p3_s50`, `p3_s75`, `q3_int4`, `q3_int3` (dev-set selection, matched budgets) |
| 8 | Aggregate: table.csv, bootstrap.json, matched_budget.csv, qualitative.csv, plots/ |


In [ ]:
# Cell 1: Drive + HF_HOME + data check
import os, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

HF_HOME = '/content/drive/MyDrive/atml_pa4_hf_cache'
DATA_ROOT = '/content/drive/MyDrive/atml-pa4-datasets/data/public_locked'
RESULTS_ROOT = '/content/drive/MyDrive/atml_pa4_results/task1'
CALIB_CACHE = '/content/calib_stats.pt'   # local disk, not Drive
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME

FILES = ['c4_calibration_locked', 'c4_development_locked', 'c4_validation_windows_locked',
         'wikitext2_test_windows_locked', 'hellaswag_val_locked']
missing = [f for f in FILES if not os.path.exists(f'{DATA_ROOT}/{f}.jsonl.gz')]
print('HF_HOME     :', HF_HOME)
print('DATA_ROOT   :', DATA_ROOT)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('PASS' if not missing else f'FAIL: missing {missing}')


In [ ]:
# Cell 2: clone / pull repo
import subprocess
REPO_URL = 'https://github.com/reemwaheed13/ATML-PA4'
REPO_DIR = '/content/ATML-PA4'
if os.path.isdir(REPO_DIR + '/.git'):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
else:
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
head = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('HEAD:', head)
print('PASS' if r.returncode == 0 and os.path.exists('task1_compression/run_task1.py') else 'FAIL')


In [ ]:
# Cell 3: pinned versions (handout: transformers==4.57.1, huggingface_hub==0.36.2)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.57.1', 'huggingface_hub==0.36.2'], capture_output=True, text=True)
print(r.stderr[-2000:] if r.returncode else 'pip OK')
v = subprocess.run([sys.executable, '-c',
                    'import transformers, huggingface_hub, torch;'
                    'print(transformers.__version__, huggingface_hub.__version__, torch.__version__, torch.cuda.is_available())'],
                   capture_output=True, text=True).stdout.split()
print('transformers, huggingface_hub, torch, cuda:', v)
ok = r.returncode == 0 and v[:2] == ['4.57.1', '0.36.2'] and v[3] == 'True'
print('PASS' if ok else 'FAIL (if transformers was already imported in this kernel, restart the runtime and rerun from cell 1)')


In [ ]:
# Cell 4: CPU unit tests
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], capture_output=True, text=True,
                   env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(r.stdout[-4000:])
print(r.stderr[-2000:])
print('PASS' if r.returncode == 0 else 'FAIL')


In [ ]:
# Cell 5: GPU sanity - fp16 condition only
import json
def run_driver(args):
    cmd = [sys.executable, '-u', '-m', 'task1_compression.run_task1'] + args
    print(' '.join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

common = ['--data-root', DATA_ROOT, '--results-root', RESULTS_ROOT, '--calib-cache', CALIB_CACHE]
rc = run_driver(['run', '--conditions', 'fp16'] + common)
s = json.load(open(f'{RESULTS_ROOT}/fp16/summary.json'))
print(f"C4 PPL  {s['c4_val']['ppl']:.4f}  (N={s['c4_val']['N']})")
print(f"WT2 PPL {s['wt2_test']['ppl']:.4f}  (N={s['wt2_test']['N']})")
print(f"HellaSwag n={s['hellaswag']['n_examples']} acc={s['hellaswag']['raw_acc']:.4f} acc_norm={s['hellaswag']['acc_norm']:.4f}")
print(f"prefill512 {s['latency']['prefill_512']['mean_ms']:.1f} ms, prefill2048 {s['latency']['prefill_2048']['mean_ms']:.1f} ms, "
      f"decode {s['latency']['decode_64_after_prefill_512']['ms_per_token']:.2f} ms/token, peak eval {s['peak_alloc_mb_eval']:.0f} MB")
ok = rc == 0 and s['c4_val']['N'] == 50000 and s['wt2_test']['N'] == 50000 and s['hellaswag']['n_examples'] == 2000
print('PASS' if ok else 'FAIL')


In [ ]:
# Cell 6: overnight run - 14 non-adaptive conditions (resumable; rerun after a disconnect)
OVERNIGHT = ['fp16', 'p1_s25', 'p1_s50', 'p1_s75', 'p2_s25', 'p2_s50', 'p2_s75', 'p2_24',
             'q1_int8', 'q1_int4', 'q1_int3', 'q2_int8', 'q2_int4', 'q2_int3']
rc = run_driver(['run', '--conditions', ','.join(OVERNIGHT)] + common)
done = [c for c in OVERNIGHT if os.path.exists(f'{RESULTS_ROOT}/{c}/summary.json')]
print(f'{len(done)}/{len(OVERNIGHT)} conditions complete; missing: {sorted(set(OVERNIGHT) - set(done))}')
print('PASS' if rc == 0 and len(done) == len(OVERNIGHT) else 'FAIL')


In [ ]:
# Cell 7: P3 / Q3 (resumable).  Sensitivities use the cached 128-seq calibration stats
# (CALIB_CACHE); lambda (P3) and b_min (Q3) are chosen on the 32 C4 development
# sequences only.  The Q3 bit-error table is cached on local disk and shared by
# q3_int4 / q3_int3.  Q2 budgets come from byte accounting (q2 need not have run).
Q3_TABLE_CACHE = '/content/q3_table.pt'
ADAPTIVE = ['p3_s50', 'p3_s75', 'q3_int4', 'q3_int3']
rc = run_driver(['run', '--conditions', ','.join(ADAPTIVE), '--q3-table-cache', Q3_TABLE_CACHE] + common)
ok = rc == 0
for c in ADAPTIVE:
    path = f'{RESULTS_ROOT}/{c}/summary.json'
    if not os.path.exists(path):
        print(f'{c}: missing summary.json'); ok = False; continue
    s = json.load(open(path))
    d = s['details']
    if c.startswith('p3'):
        p = d['p3']
        print(f"{c}: lambda={p['lambda']}  zeros {p['achieved_zeros']} vs P2 {p['p2_zero_count_same_S']}  "
              f"C4 {s['c4_val']['ppl']:.3f}  WT2 {s['wt2_test']['ppl']:.3f}  compress {s['compression_seconds']:.0f}s")
        ok &= p['matches_p2_zero_count']
    else:
        q = d['q3']; bc = q['budget_check']
        print(f"{c}: b_min={q['b_min']}  bytes {bc['q3_total_bytes']} <= budget {bc['budget_bytes_q2_uniform']}  "
              f"C4 {s['c4_val']['ppl']:.3f}  WT2 {s['wt2_test']['ppl']:.3f}  compress {s['compression_seconds']:.0f}s")
        ok &= bc['within_budget']
    sel = json.load(open(f'{RESULTS_ROOT}/{c}/{c}_selection.json'))
    print('   dev grid:', [(g.get('lam', g.get('b_min')), round(g['dev_ppl'], 3)) for g in sel['grid'] if 'dev_ppl' in g])
print('PASS' if ok else 'FAIL')


In [ ]:
# Cell 8: aggregate
rc = run_driver(['aggregate', '--results-root', RESULTS_ROOT, '--data-root', DATA_ROOT])
import csv
rows = list(csv.DictReader(open(f'{RESULTS_ROOT}/table.csv')))
for r in rows:
    print(f"{r['condition']:9s} C4 {float(r['c4_ppl']):9.3f}  WT2 {float(r['wt2_ppl']):9.3f}  "
          f"HS512 {float(r['hs512_acc_norm']):.4f}  sparsity {float(r['achieved_eligible_sparsity']):.3f}")
mb = f'{RESULTS_ROOT}/matched_budget.csv'
if os.path.exists(mb):
    for r in csv.DictReader(open(mb)):
        print(f"{r['pair']:22s} adaptive {r['adaptive']}  uniform {r['uniform_reference']}  matched {r['matched']}")
print('plots:', sorted(os.listdir(f'{RESULTS_ROOT}/plots')) if os.path.isdir(f'{RESULTS_ROOT}/plots') else None)
print('PASS' if rc == 0 and rows else 'FAIL')
